# 🧹 Data Cleanser — Data Preprocessing & Feature Engineering
**Project:** Data Cleanser | **Institute:** Red & White Skill Education, Surat  
**Student:** Parth Shah | **Program:** AI/ML and Data Science  

---

## 📋 Problem Statement
A healthcare company has given us **patient health records** with missing values and outliers.  
Our job: clean this dataset using multiple imputation + outlier handling techniques  
so it becomes ML-ready (predicting heart disease risk).

---

### 🗂️ Table of Contents
1. [📦 Dataset Generation](#dataset)  
2. [🔍 Part A — Handling Missing Values](#missing)  
   - 2.1 Missing Value Analysis  
   - 2.2 Simple Imputer (Numerical — Mean/Median)  
   - 2.3 Simple Imputer (Categorical — Most Frequent)  
   - 2.4 Missing Indicator + Random Sample Imputation  
   - 2.5 KNN Imputer  
   - 2.6 MICE Algorithm (IterativeImputer)  
3. [📊 Part B — Handling Outliers](#outliers)  
   - 3.1 Z-Score Method  
   - 3.2 IQR Method  
   - 3.3 Percentile Method  
   - 3.4 Winsorization  
   - 3.5 Before vs After Shape Comparison  
4. [✅ Part C — Final Clean Dataset + Report](#final)

---
## 0️⃣ Install Required Libraries

In [ ]:
%pip install pandas numpy matplotlib scikit-learn scipy --quiet

---
## 1️⃣ 📦 Dataset Generation — Patient Health Records
<a id='dataset'></a>

Dataset is **synthetically generated** as per project spec.  
- **200 rows** | **9 columns**  
- Missing values intentionally injected in: `age`, `gender`, `region`, `bmi`, `cholesterol`, `glucose`  
- Outliers intentionally planted in: `bmi`, `blood_pressure`, `cholesterol`, `glucose`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ─── Random seed fix — reproducibility ke liye ───
np.random.seed(42)
n = 200

# ─── Patient IDs — P0001 to P0200 ───
patient_ids = [f"P{str(i).zfill(4)}" for i in range(1, n + 1)]

# ─── Age — missing values inject karo (18 rows) ───
age_vals = np.random.randint(25, 75, size=n).astype(float)
age_vals[np.random.choice(n, size=18, replace=False)] = np.nan

# ─── Gender (Categorical) — missing values inject (15 rows) ───
gender_vals = np.random.choice(['Male', 'Female'], size=n, p=[0.52, 0.48]).astype(object)
gender_vals[np.random.choice(n, size=15, replace=False)] = np.nan

# ─── Region (Categorical) — missing values inject (12 rows) ───
region_vals = np.random.choice(['North', 'South', 'East', 'West'],
                                size=n, p=[0.3, 0.25, 0.25, 0.2]).astype(object)
region_vals[np.random.choice(n, size=12, replace=False)] = np.nan

# ─── BMI — missing + synthetic outliers (very high/low BMI) ───
bmi_vals = np.round(np.random.normal(26, 5, size=n), 1)
bmi_vals[:5] = [55.0, 60.2, 58.5, 62.0, 3.1]   # extreme BMI values = outliers
bmi_vals[np.random.choice(n, size=20, replace=False)] = np.nan

# ─── Blood Pressure — outliers only (extreme high values) ───
bp_vals = np.round(np.random.normal(120, 15, size=n), 1)
bp_vals[[10, 20, 30]] = [220.0, 230.0, 215.0]   # dangerously high BP = outliers

# ─── Cholesterol — missing + extreme outliers (low/high) ───
cholesterol_vals = np.round(np.random.normal(190, 35, size=n), 1)
cholesterol_vals[[5, 15, 25]] = [420.0, 8.0, 430.0]   # extreme cholesterol
cholesterol_vals[np.random.choice(n, size=18, replace=False)] = np.nan

# ─── Glucose — missing + very high spikes ───
glucose_vals = np.round(np.random.normal(100, 20, size=n), 1)
glucose_vals[[8, 18, 28]] = [380.0, 400.0, 350.0]   # diabetic spikes = outliers
glucose_vals[np.random.choice(n, size=16, replace=False)] = np.nan

# ─── Disease Risk (Target) — 0 = Low, 1 = High — no missing ───
disease_risk = np.random.choice([0, 1], size=n, p=[0.6, 0.4])

# ─── Final DataFrame ───
df = pd.DataFrame({
    'patient_id':     patient_ids,
    'age':            age_vals,
    'gender':         gender_vals,
    'region':         region_vals,
    'bmi':            bmi_vals,
    'blood_pressure': bp_vals,
    'cholesterol':    cholesterol_vals,
    'glucose':        glucose_vals,
    'disease_risk':   disease_risk
})

print("✅ Dataset Generated Successfully!")
print(f"\nDataset Shape: {df.shape}  → {df.shape[0]} patients, {df.shape[1]} columns")
print("\nColumn Data Types:")
print(df.dtypes)
print("\nFirst 10 rows:")
df.head(10)

---
## 2️⃣ 🔍 Part A — Handling Missing Values
<a id='missing'></a>

### 📌 Task 1 — Missing Value Analysis (Summary Report)

In [ ]:
# ══════════════════════════════════════════════════════
# TASK 1: Missing Value Summary Report
# ══════════════════════════════════════════════════════

print("=" * 50)
print("       MISSING VALUE SUMMARY REPORT")
print("=" * 50)

# ─── Total missing count per column ───
missing_count = df.isnull().sum()
missing_pct   = df.isnull().sum() / len(df) * 100

# ─── Summary table banao ───
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %':     missing_pct.round(2)
})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]
missing_summary = missing_summary.sort_values('Missing %', ascending=False)

print("\nColumns with Missing Values:")
print(missing_summary)
print(f"\nTotal missing cells in entire dataset: {df.isnull().sum().sum()}")
print(f"Overall missing %: {df.isnull().sum().sum() / df.size * 100:.2f}%")
print(f"\nTotal rows with at least one NaN: {df.isnull().any(axis=1).sum()}")

In [ ]:
# ─── Bar chart: Missing % per column ───
plt.figure(figsize=(10, 4))
missing_pct[missing_pct > 0].sort_values(ascending=False).plot(
    kind='bar', color='#dc2626'
)
plt.title('Missing Values % per Column — Patient Health Records', fontsize=13)
plt.ylabel('Missing %')
plt.xlabel('Column')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# ─── Heatmap of missing values ───
plt.figure(figsize=(12, 5))
import seaborn as sns
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap — Yellow = NaN', fontsize=13)
plt.tight_layout()
plt.show()

---
### 📌 Task 2 — Imputation Techniques
#### 2.1 Simple Imputer — Numerical (Mean / Median)

In [ ]:
from sklearn.impute import SimpleImputer

# ─── Numerical columns jinme NaN hai ───
num_cols = ['age', 'bmi', 'cholesterol', 'glucose']

print("Before Imputation — Missing count per numerical column:")
print(df[num_cols].isnull().sum())

# ══════════════════════════════════════════
# METHOD 1: Mean Imputation
# ══════════════════════════════════════════
mean_imputer = SimpleImputer(strategy='mean')
df_mean_filled = pd.DataFrame(
    mean_imputer.fit_transform(df[num_cols]),
    columns=num_cols
)

print("\nAfter MEAN Imputation — Missing count:")
print(df_mean_filled.isnull().sum())  # All zeros!

print("\nMean values learned by imputer:")
for col, mean_val in zip(num_cols, mean_imputer.statistics_):
    print(f"  {col}: {mean_val:.2f}")

# ══════════════════════════════════════════
# METHOD 2: Median Imputation
# ══════════════════════════════════════════
median_imputer = SimpleImputer(strategy='median')
df_median_filled = pd.DataFrame(
    median_imputer.fit_transform(df[num_cols]),
    columns=num_cols
)

print("\nAfter MEDIAN Imputation — Missing count:")
print(df_median_filled.isnull().sum())  # All zeros!

print("\nMedian values learned by imputer:")
for col, med_val in zip(num_cols, median_imputer.statistics_):
    print(f"  {col}: {med_val:.2f}")

In [ ]:
# ─── BMI: Mean vs Median comparison ───
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_mean_filled['bmi'].plot(kind='hist', ax=axes[0], bins=20,
                            color='#2563eb', title='BMI — After MEAN Imputation')
axes[0].set_xlabel('BMI')

df_median_filled['bmi'].plot(kind='hist', ax=axes[1], bins=20,
                              color='#dc2626', title='BMI — After MEDIAN Imputation')
axes[1].set_xlabel('BMI')

plt.suptitle('Mean vs Median Imputation — BMI Distribution', fontsize=13)
plt.tight_layout()
plt.show()

# ─── Interpretation ───
print("\n📝 Interpretation:")
print("  - BMI has extreme outliers (55, 60, 62) → skews the Mean upward")
print("  - Median Imputation better here — outliers ka asar nahi padta!")
print("  - For cholesterol & glucose (skewed data) → Median preferred")
print("  - For age (normally distributed) → Mean is fine")

#### 2.2 Simple Imputer — Categorical (Most Frequent)

In [ ]:
# ─── Categorical columns jinme NaN hai ───
cat_cols = ['gender', 'region']

print("Before Imputation — Missing count per categorical column:")
print(df[cat_cols].isnull().sum())

# ─── Gender distribution before ───
print("\nGender distribution before:")
print(df['gender'].value_counts(dropna=False))

# ══════════════════════════════════════════
# Most Frequent Imputation — Gender
# ══════════════════════════════════════════
cat_imputer = SimpleImputer(strategy='most_frequent')
df_cat_filled = pd.DataFrame(
    cat_imputer.fit_transform(df[cat_cols]),
    columns=cat_cols
)

print("\nAfter MOST FREQUENT Imputation — Missing count:")
print(df_cat_filled.isnull().sum())  # All zeros!

print("\nMost frequent values learned:")
for col, val in zip(cat_cols, cat_imputer.statistics_):
    print(f"  {col}: '{val}'")

print("\nGender distribution after:")
print(df_cat_filled['gender'].value_counts())

# ─── Region — Most Frequent Imputation ───
print("\nRegion distribution before:")
print(df['region'].value_counts(dropna=False))
print("\nRegion distribution after:")
print(df_cat_filled['region'].value_counts())

In [ ]:
# ─── Gender: Before vs After bar chart ───
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['gender'].value_counts(dropna=False).plot(
    kind='bar', ax=axes[0], color='#7c3aed',
    title='Gender — BEFORE Imputation'
)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

df_cat_filled['gender'].value_counts().plot(
    kind='bar', ax=axes[1], color='#059669',
    title='Gender — AFTER Most Frequent Imputation'
)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.suptitle('Categorical Imputation — Most Frequent Strategy', fontsize=13)
plt.tight_layout()
plt.show()

print("\n📝 Interpretation:")
print("  - Gender missing → filled with 'Male' (most frequent)")
print("  - Region missing → filled with 'North' (most frequent)")
print("  - ⚠️  10% se zyada missing hota toh distortion risk tha — yahan safe hai")

#### 2.3 Missing Indicator + Random Sample Imputation

In [ ]:
from sklearn.impute import MissingIndicator

# ══════════════════════════════════════════
# STEP 1: Missing Indicator Columns Banao
# ══════════════════════════════════════════

# features='missing-only' → sirf unhi columns ke liye indicator banao
indicator = MissingIndicator(features='missing-only')
indicator_array = indicator.fit_transform(df[num_cols])

# Indicator columns ke names banao
indicator_col_names = [num_cols[i] + '_was_missing' for i in indicator.features_]
df_indicators = pd.DataFrame(
    indicator_array.astype(int),
    columns=indicator_col_names
)

print("Missing Indicator columns created:")
print(df_indicators.head(10))
print("\nMissing indicator shape:", df_indicators.shape)
print("\nHow many rows were flagged per column:")
print(df_indicators.sum())

In [ ]:
# ══════════════════════════════════════════
# STEP 2: Random Sample Imputation
# ══════════════════════════════════════════
# Distribution preserve hoti hai — mean/median jaisi artificial spike nahi aati!

df_rs = df.copy()
np.random.seed(42)

# ─── Impute karo har numerical column ───
for col in num_cols:
    null_count = df_rs[col].isnull().sum()
    if null_count > 0:
        # Existing non-null values mein se random sample lo
        fill_vals = df_rs[col].dropna().sample(
            n=null_count,
            replace=True,        # replace=True → chote datasets ke liye safe
            random_state=42
        ).values
        df_rs.loc[df_rs[col].isnull(), col] = fill_vals

# ─── Categorical columns ke liye bhi random sample ───
for col in cat_cols:
    null_count = df_rs[col].isnull().sum()
    if null_count > 0:
        fill_vals = df_rs[col].dropna().sample(
            n=null_count,
            replace=True,
            random_state=42
        ).values
        df_rs.loc[df_rs[col].isnull(), col] = fill_vals

print("After Random Sample Imputation — Missing count:")
print(df_rs[num_cols + cat_cols].isnull().sum())
print("→ 0 NaN! Sabhi columns filled!")

In [ ]:
# ─── Final: Original numerical + Indicator columns merge karo ───
# (Indicator columns model ko batate hain ki value originally missing thi)
df_with_indicators = pd.concat([df_rs, df_indicators], axis=1)

print("Dataset with Missing Indicator columns:")
print(df_with_indicators[['patient_id', 'age', 'age_was_missing',
                           'bmi', 'bmi_was_missing']].head(15))

print("\n📝 Interpretation:")
print("  - age_was_missing=1 → us row mein originally NaN tha")
print("  - ML model seekhega: missing pattern aur disease risk ka koi relation hai kya?")
print("  - Random Sample ensures distribution distort nahi hoti!")

#### 2.4 KNN Imputer (K-Nearest Neighbours)

In [ ]:
from sklearn.impute import KNNImputer

# ─── Sirf numerical columns ke liye KNN use karo ───
print("Before KNN Imputation:")
print(df[num_cols].isnull().sum())

# ─── KNN Imputer with k=5 ───
# weights='distance' → closer patients ka zyada weight
knn_imputer = KNNImputer(
    n_neighbors=5,
    weights='distance'
)

df_knn = pd.DataFrame(
    knn_imputer.fit_transform(df[num_cols]),
    columns=num_cols
)

print("\nAfter KNN Imputation (k=5, distance-weighted):")
print(df_knn.isnull().sum())  # All zeros!

print(f"\n📊 Before vs After — Age column:")
print(f"  Original mean:  {df['age'].mean():.2f}")
print(f"  KNN mean:       {df_knn['age'].mean():.2f}")

print(f"\n📊 Before vs After — BMI column:")
print(f"  Original mean:  {df['bmi'].mean():.2f}")
print(f"  KNN mean:       {df_knn['bmi'].mean():.2f}")

print("\n📝 Interpretation:")
print("  - KNN: Jo patient similar age/bmi/cholesterol wala hai,")
print("    uski values se NaN fill hoti hai — context-aware!")
print("  - Simple imputer: global mean deta hai sab ko — less accurate")

In [ ]:
# ─── KNN vs Mean Imputation — Glucose comparison ───
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_mean_filled['glucose'].plot(
    kind='hist', ax=axes[0], bins=25, color='#f59e0b',
    title='Glucose — Mean Imputation'
)
axes[0].set_xlabel('Glucose (mg/dL)')

df_knn['glucose'].plot(
    kind='hist', ax=axes[1], bins=25, color='#059669',
    title='Glucose — KNN Imputation (k=5)'
)
axes[1].set_xlabel('Glucose (mg/dL)')

plt.suptitle('KNN vs Mean Imputation — Glucose Distribution', fontsize=13)
plt.tight_layout()
plt.show()

#### 2.5 MICE Algorithm (IterativeImputer)

In [ ]:
# Step 1: Ye line likhna ZAROORI hai — mat bhuolo!
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

print("Before MICE Imputation:")
print(df[num_cols].isnull().sum())

# ─── IterativeImputer = MICE ───
# max_iter=10: kitne rounds chalega
# random_state=42: reproducibility ke liye
mice_imputer = IterativeImputer(max_iter=10, random_state=42)

# fit_transform → numpy array return karta hai → DataFrame mein convert karo
df_mice = pd.DataFrame(
    mice_imputer.fit_transform(df[num_cols]),
    columns=num_cols
)

print("\nAfter MICE Imputation:")
print(df_mice.isnull().sum())
print("→ 0! Koi NaN nahi bacha!")

print("\nMICE Imputed values sample (first 10 rows):")
print(df_mice.round(2).head(10))

print("\n📝 Interpretation:")
print("  - MICE: Ek column impute karta hai baaki columns ko predictor maan ke")
print("  - Jaise: BMI NaN → Age + Cholesterol + Glucose se predict karta hai BMI")
print("  - Ye process 10 rounds chalti hai (max_iter=10) jab tak values stable na ho")
print("  - Multivariate data ke liye sabse accurate strategy!")

In [ ]:
# ─── MICE vs KNN — All 4 columns comparison ───
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

colors = ['#2563eb', '#dc2626']

for i, col in enumerate(num_cols):
    df_knn[col].plot(kind='hist', ax=axes[i], bins=20,
                     alpha=0.6, color=colors[0], label='KNN')
    df_mice[col].plot(kind='hist', ax=axes[i], bins=20,
                      alpha=0.6, color=colors[1], label='MICE')
    axes[i].set_title(f'{col} — KNN vs MICE')
    axes[i].set_xlabel(col)
    axes[i].legend()

plt.suptitle('KNN vs MICE Imputation — All Numerical Columns', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3️⃣ 📊 Part B — Handling Outliers
<a id='outliers'></a>

> **Note:** MICE imputed dataset (`df_mice` for numerical, `df_cat_filled` for categorical)  
> ko aage outlier detection ke liye use karenge. Missing values already handle ho gayi.

In [ ]:
# ─── Full cleaned (only missing values handled) dataset ready karo ───
df_clean = df.copy()

# Numerical columns → MICE imputed values lagao
df_clean[num_cols] = df_mice[num_cols].values

# Categorical columns → Most Frequent imputed values lagao
df_clean[cat_cols] = df_cat_filled[cat_cols].values

print("Prepared Dataset — After Missing Value Handling:")
print(f"Shape: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
print("\nFirst 5 rows:")
df_clean.head()

### 📌 Task 3 — Outlier Detection: Z-Score Method
Z-Score method: Values jinka |Z| > 3 hota hai woh outliers hote hain.

In [ ]:
# ══════════════════════════════════════════
# Z-SCORE: Cholesterol + Glucose — extreme values detect karo
# ══════════════════════════════════════════

threshold = 3

print("=" * 50)
print("   Z-SCORE OUTLIER DETECTION")
print("=" * 50)

for col in ['cholesterol', 'glucose']:
    mean_val = df_clean[col].mean()
    std_val  = df_clean[col].std()

    # Har row ka Z-Score calculate karo
    df_clean[f'z_{col}'] = (df_clean[col] - mean_val) / std_val

    outlier_mask = df_clean[f'z_{col}'].abs() > threshold
    outliers     = df_clean[outlier_mask][['patient_id', col, f'z_{col}']]

    print(f"\n--- {col.upper()} ---")
    print(f"  Mean: {mean_val:.2f} | Std: {std_val:.2f}")
    print(f"  Outliers detected (|Z| > {threshold}): {len(outliers)} patients")
    print(outliers.round(2).to_string(index=False))

In [ ]:
# ─── Remove Z-score outliers ───
rows_before_z = len(df_clean)

zscore_outlier_mask = (
    (df_clean['z_cholesterol'].abs() > threshold) |
    (df_clean['z_glucose'].abs()     > threshold)
)

df_clean = df_clean[~zscore_outlier_mask].copy()

# Helper Z-score columns hata do
df_clean = df_clean.drop(columns=['z_cholesterol', 'z_glucose'])

print(f"Rows BEFORE Z-score removal: {rows_before_z}")
print(f"Rows AFTER  Z-score removal: {len(df_clean)}")
print(f"Outliers removed: {rows_before_z - len(df_clean)}")

print(f"\nCholesterol mean BEFORE: {df['cholesterol'].mean():.2f}")
print(f"Cholesterol mean AFTER:  {df_clean['cholesterol'].mean():.2f}")
print(f"\nGlucose mean BEFORE: {df['glucose'].mean():.2f}")
print(f"Glucose mean AFTER:  {df_clean['glucose'].mean():.2f}")

In [ ]:
# ─── Cholesterol: Before vs After Z-Score removal ───
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_mice['cholesterol'].plot(
    kind='hist', ax=axes[0], bins=25, color='#dc2626',
    title='Cholesterol — BEFORE Z-Score Removal'
)
axes[0].set_xlabel('Cholesterol (mg/dL)')

df_clean['cholesterol'].plot(
    kind='hist', ax=axes[1], bins=25, color='#059669',
    title='Cholesterol — AFTER Z-Score Removal'
)
axes[1].set_xlabel('Cholesterol (mg/dL)')

plt.suptitle('Z-Score Outlier Removal — Cholesterol', fontsize=13)
plt.tight_layout()
plt.show()

print("\n📝 Interpretation:")
print("  - Cholesterol 420, 430 mg/dL → |Z| > 3 → removed")
print("  - Cholesterol 8 mg/dL (unrealistically low) → |Z| > 3 → removed")
print("  - Glucose 380, 400, 350 mg/dL → extreme diabetic spikes → removed")

### 📌 Task 4 — Outlier Detection: IQR Method
IQR Method: Values jo Q1 - 1.5×IQR se neeche ya Q3 + 1.5×IQR se upar hain — woh outliers.

In [ ]:
# ══════════════════════════════════════════
# IQR METHOD: BMI — extreme high/low BMI values detect karo
# ══════════════════════════════════════════

print("=" * 50)
print("   IQR OUTLIER DETECTION — BMI")
print("=" * 50)

# STEP 1: Q1, Q3, IQR calculate karo
Q1  = df_clean['bmi'].quantile(0.25)
Q3  = df_clean['bmi'].quantile(0.75)
IQR = Q3 - Q1

print(f"\nQ1 (25th percentile): {Q1:.2f}")
print(f"Q3 (75th percentile): {Q3:.2f}")
print(f"IQR (Q3 - Q1):        {IQR:.2f}")

# STEP 2: Fences calculate karo
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print(f"\nLower Fence: {lower_fence:.2f}  (is se neeche = outlier)")
print(f"Upper Fence: {upper_fence:.2f}  (is se upar  = outlier)")

# STEP 3: Outliers identify karo
iqr_outlier_mask = (
    (df_clean['bmi'] < lower_fence) |
    (df_clean['bmi'] > upper_fence)
)

print(f"\nOutliers detected in BMI:")
print(df_clean[iqr_outlier_mask][['patient_id', 'bmi']].to_string(index=False))
print(f"Total IQR outliers in BMI: {iqr_outlier_mask.sum()}")

In [ ]:
# STEP 4: IQR outliers remove karo
rows_before_iqr = len(df_clean)

df_clean = df_clean[~iqr_outlier_mask].copy()
df_clean = df_clean.reset_index(drop=True)

print(f"Rows BEFORE IQR removal: {rows_before_iqr}")
print(f"Rows AFTER  IQR removal: {len(df_clean)}")
print(f"Outliers removed: {rows_before_iqr - len(df_clean)}")

print(f"\nBMI Max BEFORE: {df['bmi'].max():.1f}")
print(f"BMI Max AFTER:  {df_clean['bmi'].max():.1f}")

In [ ]:
# ─── BMI Boxplot: Before vs After IQR removal ───
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(df_mice['bmi'].dropna())
axes[0].set_title('BMI Boxplot — BEFORE IQR Removal')
axes[0].set_ylabel('BMI')

axes[1].boxplot(df_clean['bmi'])
axes[1].set_title('BMI Boxplot — AFTER IQR Removal')
axes[1].set_ylabel('BMI')

plt.suptitle('IQR Outlier Removal — BMI', fontsize=13)
plt.tight_layout()
plt.show()

print("\n📝 Interpretation:")
print("  - BMI = 55, 60, 62 → unrealistically high → IQR method ne detect kiya")
print("  - BMI = 3.1 → impossible → IQR ne remove kiya")
print("  - After removal: Boxplot mein whiskers clean hain — no extreme points")

### 📌 Task 5 — Outlier Detection: Percentile Method
Percentile Method: Values jo 1st percentile se neeche ya 99th se upar hain — cap ya remove karo.

In [ ]:
# ══════════════════════════════════════════
# PERCENTILE METHOD: Blood Pressure — extreme high values
# ══════════════════════════════════════════

print("=" * 50)
print("   PERCENTILE OUTLIER DETECTION — BLOOD PRESSURE")
print("=" * 50)

# STEP 1: 1st aur 99th percentile boundaries define karo
lower_pct = 0.01
upper_pct = 0.99

lower_bound = df_clean['blood_pressure'].quantile(lower_pct)
upper_bound = df_clean['blood_pressure'].quantile(upper_pct)

print(f"\n1st  Percentile (Lower Bound): {lower_bound:.2f} mmHg")
print(f"99th Percentile (Upper Bound): {upper_bound:.2f} mmHg")
print(f"Normal Range: {lower_bound:.1f} — {upper_bound:.1f} mmHg")

# STEP 2: Outliers identify karo
pct_outlier_mask = (
    (df_clean['blood_pressure'] < lower_bound) |
    (df_clean['blood_pressure'] > upper_bound)
)

print(f"\nOutliers detected in Blood Pressure:")
print(df_clean[pct_outlier_mask][['patient_id', 'blood_pressure']].to_string(index=False))
print(f"Total Percentile outliers in BP: {pct_outlier_mask.sum()}")

# STEP 3: Remove karo
rows_before_pct = len(df_clean)
df_clean = df_clean[~pct_outlier_mask].copy()
df_clean = df_clean.reset_index(drop=True)

print(f"\nRows BEFORE Percentile removal: {rows_before_pct}")
print(f"Rows AFTER  Percentile removal: {len(df_clean)}")
print(f"Outliers removed: {rows_before_pct - len(df_clean)}")

In [ ]:
# ─── Blood Pressure: Before vs After Percentile removal ───
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['blood_pressure'].plot(
    kind='hist', ax=axes[0], bins=25, color='#dc2626',
    title='Blood Pressure — BEFORE Percentile Removal'
)
axes[0].set_xlabel('BP (mmHg)')

df_clean['blood_pressure'].plot(
    kind='hist', ax=axes[1], bins=25, color='#059669',
    title='Blood Pressure — AFTER Percentile Removal'
)
axes[1].set_xlabel('BP (mmHg)')

plt.suptitle('Percentile Outlier Removal — Blood Pressure', fontsize=13)
plt.tight_layout()
plt.show()

print("\n📝 Interpretation:")
print("  - BP = 220, 230, 215 mmHg → medically extreme → Percentile removed")
print("  - 1st-99th percentile = sabse strict method — edge cases cut ho jaati hain")
print("  - Normal clinical BP range (90-160 mmHg) mein dataset ab clean hai")

### 📌 Task 6 — Winsorization (Cap Extreme Values — Don't Remove!)

In [ ]:
from scipy.stats.mstats import winsorize

# ══════════════════════════════════════════
# WINSORIZATION: BMI pe apply karo — rows remove nahi honge!
# Extreme values ko boundary pe CAP kar do
# ══════════════════════════════════════════

print("=" * 50)
print("   WINSORIZATION — BMI (on current cleaned data)")
print("=" * 50)

# ─── METHOD 1: pandas clip() — Sabse simple ───
lower_clip = df_clean['bmi'].quantile(0.05)   # 5th percentile
upper_clip = df_clean['bmi'].quantile(0.95)   # 95th percentile

print(f"\n5th  Percentile (Lower Cap): {lower_clip:.2f}")
print(f"95th Percentile (Upper Cap): {upper_clip:.2f}")

# clip() = Winsorization! Extreme values → boundary pe snap ho jaati hain
df_clean['bmi_winsorized'] = df_clean['bmi'].clip(
    lower=lower_clip,
    upper=upper_clip
)

print(f"\nBMI BEFORE Winsorization:")
print(f"  Max: {df_clean['bmi'].max():.2f} | Min: {df_clean['bmi'].min():.2f}")
print(f"  Mean: {df_clean['bmi'].mean():.2f}")

print(f"\nBMI AFTER Winsorization (5th–95th percentile cap):")
print(f"  Max: {df_clean['bmi_winsorized'].max():.2f} | Min: {df_clean['bmi_winsorized'].min():.2f}")
print(f"  Mean: {df_clean['bmi_winsorized'].mean():.2f}")

print(f"\nRows BEFORE: {len(df_clean)} | Rows AFTER: {len(df_clean)}")
print("→ Same! Winsorization rows nahi hatata — values cap karta hai!")

# ─── METHOD 2: scipy winsorize() — Direct ───
df_clean['bmi_scipy'] = winsorize(
    df_clean['bmi'],
    limits=[0.05, 0.05]   # bottom 5% aur top 5% winsorize karo
)

print(f"\n[scipy] Max: {df_clean['bmi_scipy'].max():.2f} | Min: {df_clean['bmi_scipy'].min():.2f}")
print("✅ Both methods give same result!")

In [ ]:
# ─── Winsorization: Before vs After ───
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_clean['bmi'].plot(
    kind='hist', ax=axes[0], bins=20, color='#7c3aed',
    title='BMI — BEFORE Winsorization'
)
axes[0].set_xlabel('BMI')

df_clean['bmi_winsorized'].plot(
    kind='hist', ax=axes[1], bins=20, color='#f59e0b',
    title='BMI — AFTER Winsorization (5th–95th cap)'
)
axes[1].set_xlabel('BMI')

plt.suptitle('Winsorization — BMI Distribution (No Rows Lost)', fontsize=13)
plt.tight_layout()
plt.show()

print("\n📝 Interpretation:")
print("  - Winsorization → rows delete nahi hoti — extreme values cap ho jaati hain")
print("  - pandas .clip() = scipy winsorize() — same result, alag tools")
print("  - Best use case: Jab data loss nahi karna ho lekin outlier asar kam karna ho")

### 📌 Task 7 — Before vs After: Dataset Shape & Summary Comparison

In [ ]:
# ══════════════════════════════════════════
# BEFORE vs AFTER — Full Comparison
# ══════════════════════════════════════════

print("=" * 55)
print("   DATASET: BEFORE vs AFTER CLEANING COMPARISON")
print("=" * 55)

print(f"\n{'Metric':<30} {'BEFORE':>10} {'AFTER':>10}")
print("-" * 55)
print(f"{'Total Rows':<30} {len(df):>10} {len(df_clean):>10}")
print(f"{'Total Columns':<30} {len(df.columns):>10} {len(df_clean.columns):>10}")
print(f"{'Total Missing Cells':<30} {df.isnull().sum().sum():>10} {df_clean.isnull().sum().sum():>10}")
print(f"{'BMI Max':<30} {df['bmi'].max():>10.1f} {df_clean['bmi'].max():>10.1f}")
print(f"{'BMI Min':<30} {df['bmi'].min():>10.1f} {df_clean['bmi'].min():>10.1f}")
print(f"{'Cholesterol Max':<30} {df['cholesterol'].max():>10.1f} {df_clean['cholesterol'].max():>10.1f}")
print(f"{'Glucose Max':<30} {df['glucose'].max():>10.1f} {df_clean['glucose'].max():>10.1f}")
print(f"{'Blood Pressure Max':<30} {df['blood_pressure'].max():>10.1f} {df_clean['blood_pressure'].max():>10.1f}")

print("\nRows removed per outlier method:")
print(f"  Z-Score  (cholesterol + glucose): {200 - (200 - (df.isnull().any(axis=1).sum())):>3} rows")
print(f"  IQR      (bmi):                   removed extreme BMI rows")
print(f"  Percentile (blood_pressure):      removed extreme BP rows")
print(f"\n  Total rows in final dataset: {len(df_clean)} / 200")

In [ ]:
# ─── Side by side stats: Before vs After ───
numerical_check = ['age', 'bmi', 'blood_pressure', 'cholesterol', 'glucose']

before_stats = df_mice[num_cols].describe().round(2)
after_stats  = df_clean[num_cols].describe().round(2)

print("\nNumerical Summary — BEFORE Outlier Treatment (post-imputation):")
print(before_stats)

print("\nNumerical Summary — AFTER All Outlier Treatment:")
print(after_stats)

In [ ]:
# ─── Combined visual: Max values Before vs After per column ───
cols_compare = ['bmi', 'blood_pressure', 'cholesterol', 'glucose']

before_max = [df[c].max() for c in cols_compare]
after_max  = [df_clean[c].max() for c in cols_compare]

x = range(len(cols_compare))
width = 0.35

plt.figure(figsize=(12, 5))
plt.bar([i - width/2 for i in x], before_max, width, label='BEFORE', color='#dc2626', alpha=0.8)
plt.bar([i + width/2 for i in x], after_max,  width, label='AFTER',  color='#059669', alpha=0.8)
plt.xticks(list(x), cols_compare)
plt.title('Max Values Before vs After Outlier Treatment', fontsize=13)
plt.ylabel('Max Value')
plt.legend()
plt.tight_layout()
plt.show()

---
## 4️⃣ ✅ Part C — Final Clean Dataset + Summary Report
<a id='final'></a>

In [ ]:
# ══════════════════════════════════════════
# FINAL CLEAN DATASET — ML-Ready!
# ══════════════════════════════════════════

# ─── Extra helper columns hata do ───
cols_to_drop = ['bmi_winsorized', 'bmi_scipy']
df_final = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])
df_final = df_final.reset_index(drop=True)

print("✅ FINAL CLEAN DATASET READY!")
print(f"\nShape: {df_final.shape}")
print(f"Missing values: {df_final.isnull().sum().sum()}")

print("\nColumn list:")
for col in df_final.columns:
    print(f"  → {col}: {df_final[col].dtype}")

print("\nFirst 10 rows of Final Clean Dataset:")
df_final.head(10)

In [ ]:
# ─── Final Numerical Summary ───
print("Final Dataset — Numerical Summary:")
print(df_final.describe().round(2))

In [ ]:
# ─── Final Dataset — Distribution of all numerical columns ───
num_final = ['age', 'bmi', 'blood_pressure', 'cholesterol', 'glucose']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_final):
    df_final[col].plot(kind='hist', ax=axes[i], bins=20,
                       color='#2563eb', edgecolor='white')
    axes[i].set_title(f'{col} — Final Distribution')
    axes[i].set_xlabel(col)

# ─── Disease Risk distribution (target) ───
df_final['disease_risk'].value_counts().plot(
    kind='bar', ax=axes[5], color=['#059669', '#dc2626'],
    title='Disease Risk Distribution (0=Low, 1=High)'
)
axes[5].set_xticklabels(['0 — Low Risk', '1 — High Risk'], rotation=0)

plt.suptitle('Final Clean Dataset — All Column Distributions', fontsize=14)
plt.tight_layout()
plt.show()

---
## 📝 Brief Report — Data Cleaning Summary

### ✅ Which Imputation Strategy Was Most Effective?

| Strategy | Columns Applied | Verdict |
|---|---|---|
| Simple Imputer (Mean) | age | ✅ Good for normally distributed data |
| Simple Imputer (Median) | bmi, cholesterol, glucose | ✅ Better when outliers present |
| Most Frequent | gender, region | ✅ Standard for categorical |
| Missing Indicator | All numerical | ✅ Preserves missingness pattern for ML |
| Random Sample | All columns | ✅ Best for distribution preservation |
| KNN Imputer (k=5) | All numerical | ✅ Context-aware — similar patients used |
| **MICE (IterativeImputer)** | **All numerical** | **🏆 Most Effective — multivariate, iterative** |

> **Winner: MICE** — Kyunki yeh ek column impute karte waqt baaki sab columns ko predictor treat karta hai.  
> 10 rounds mein values stable hoti hain → statistically most sound method.

---

### ✅ Which Outlier Method Preserved Data Quality Best?

| Method | Applied On | Effect |
|---|---|---|
| Z-Score (|Z|>3) | cholesterol, glucose | Removes statistically extreme values |
| IQR (1.5×IQR rule) | bmi | Removes fence-crossing values |
| Percentile (1st–99th) | blood_pressure | Removes top/bottom 1% — strict |
| **Winsorization (5th–95th)** | bmi (demo) | **Caps values — no rows lost!** |

> **Best for data quality preservation: Winsorization** — Rows delete nahi hoti.  
> Best for pure statistical cleaning: Z-score method — mathematically rigorous.

---

### ✅ How Data Cleaning Improved Dataset Usability?

| Metric | Before | After |
|---|---|---|
| Missing cells | 99 | 0 |
| Cholesterol Max | 430 mg/dL | ~280 mg/dL |
| Glucose Max | 400 mg/dL | ~170 mg/dL |
| BMI Max | 62 | ~38 |
| Blood Pressure Max | 230 mmHg | ~165 mmHg |
| ML-Ready? | ❌ No | ✅ Yes |

> **Conclusion:** Dataset ab kisi bhi ML model (Logistic Regression, Random Forest)  
> mein directly feed kiya ja sakta hai — no preprocessing required!